
<div style="border:1px solid #d9d9d9; padding:18px 20px; border-radius:10px; background:#fafafa;">
  <div style="font-size:28px; font-weight:700; margin-bottom:8px;">CMAscan</div>
  <div style="font-size:18px; font-weight:600; margin-bottom:8px;">Sequence and structural features validation</div>
  <div style="font-size:14px; line-height:1.5;">
    This notebook validates the current CMAscan canonical motif dataset by comparing sequence-based scores
    (<code>cPSSM</code> and <code>ePSSM</code>) and structure-context features (<code>IUPred3</code> long mode and <code>SASA</code>)
    between curated positive motifs, curated putative-positive motifs, and matched negative motifs from the same proteins.
    <br><br>
    <b>Quick start:</b> run the notebook with <code>Runtime -&gt; Run all</code>.
    The notebook rebuilds the validation sets from the current <code>CMAscan</code> dataset and writes tables to <code>Results/</code>
    and figures to <code>Results/Figures/</code>.
    <br><br>
    <b>Scope:</b> this version uses only <code>cPSSM</code>, <code>ePSSM</code>, <code>IUPred3</code>, and <code>SASA</code>.
    Legacy charge-based and auxiliary structure metrics are intentionally excluded.
  </div>
</div>


In [ ]:

#@title Import libraries

import sys
!{sys.executable} -m pip install -q biopython openpyxl seaborn requests pandas numpy matplotlib scipy statsmodels

from io import StringIO
from pathlib import Path
import os
import pickle
import re
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from Bio import SeqIO
from IPython.display import display
import scipy.stats as stats
from statsmodels.stats.power import TTestIndPower, TTestPower, NormalIndPower

REPO_OWNER = 'Marcin-Lubocki'
REPO_NAME = 'CMAscan'
RAW_BASE_URL_CANDIDATES = [
    f'https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/main',
    f'https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/master',
]

BASE_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
REPO_ROOT = Path.cwd()
REQUEST_TIMEOUT = 120
RESULTS_DIR = REPO_ROOT / 'Results'
FIGURES_DIR = RESULTS_DIR / 'Figures'
CACHE_DIR = RESULTS_DIR / '_cache'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_WIDTH_PRESETS = {
    'three_column': 2.7,
    'two_column': 4.0,
    'full_width': 8.5,
}


def iter_repo_sources(relative_path):
    local_path = REPO_ROOT / relative_path
    if local_path.exists():
        yield str(local_path)
    for base_url in RAW_BASE_URL_CANDIDATES:
        yield f'{base_url}/{relative_path}'


def fetch_repo_file(relative_path, destination):
    destination = Path(destination)
    errors = []

    for source in iter_repo_sources(relative_path):
        try:
            if str(source).startswith('http'):
                response = requests.get(source, timeout=REQUEST_TIMEOUT)
                response.raise_for_status()
                destination.write_bytes(response.content)
            else:
                shutil.copy(source, destination)
            return destination
        except Exception as exc:  # noqa: BLE001
            errors.append(f'{source} -> {exc}')

    raise RuntimeError('Failed to fetch file: ' + '; '.join(errors))


def get_repo_local_copy(relative_path):
    local_path = REPO_ROOT / relative_path
    if local_path.exists():
        return local_path

    target = CACHE_DIR / Path(relative_path).name
    return fetch_repo_file(relative_path, target)


def get_publication_figsize(default_width, default_height, width_mode='full_width', fig_height=None):
    if width_mode not in FIGURE_WIDTH_PRESETS:
        raise ValueError(
            f"Unsupported width_mode: {width_mode}. Use one of {tuple(FIGURE_WIDTH_PRESETS)}"
        )

    fig_width = FIGURE_WIDTH_PRESETS[width_mode]
    if fig_height is None:
        fig_height = fig_width * (default_height / default_width)

    return fig_width, fig_height


def normalise_save_path(save_path, default_extension='svg'):
    save_path = str(save_path)
    root, ext = os.path.splitext(save_path)
    if ext:
        return root + '.' + default_extension
    return save_path + '.' + default_extension


def _find_arial_font_path():
    for name in ['Arial', 'ArialMT']:
        try:
            return fm.findfont(name, fallback_to_default=False)
        except Exception:
            continue

    candidate_paths = [
        Path('/usr/share/fonts/truetype/msttcorefonts/Arial.ttf'),
        Path('/usr/share/fonts/truetype/msttcorefonts/arial.ttf'),
        Path('/Library/Fonts/Arial.ttf'),
        Path('/System/Library/Fonts/Supplemental/Arial.ttf'),
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            return str(candidate)

    return None


def ensure_arial_font():
    arial_path = _find_arial_font_path()
    if arial_path:
        try:
            fm.fontManager.addfont(arial_path)
        except Exception:
            pass
        return arial_path

    if BASE_DIR == Path('/content'):
        print('Arial not found. Attempting to install Microsoft core fonts in Colab...')
        install_commands = [
            'apt-get -qq update',
            'apt-get -qq install -y software-properties-common fontconfig cabextract xfonts-utils || true',
            'add-apt-repository -y multiverse || true',
            "echo 'ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true' | debconf-set-selections || true",
            "echo 'ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula seen true' | debconf-set-selections || true",
            'DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ttf-mscorefonts-installer || true',
            'fc-cache -f -v >/dev/null 2>&1 || true',
        ]
        for command in install_commands:
            subprocess.run(command, shell=True, check=False)

        fm._load_fontmanager(try_read_cache=False)
        arial_path = _find_arial_font_path()
        if arial_path:
            try:
                fm.fontManager.addfont(arial_path)
            except Exception:
                pass
            print(f'Arial font loaded from: {arial_path}')
            return arial_path

    raise RuntimeError(
        'Arial font is required for publication figures, but it was not found. '
        'Install Arial or rerun this notebook in an environment where Arial is available.'
    )


In [ ]:

#@title Publication figure settings

ARIAL_FONT_PATH = ensure_arial_font()
print(f'Using Arial font from: {ARIAL_FONT_PATH}')

mpl.rcParams.update({
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
    'font.size': 9,
    'axes.linewidth': 0.75,
    'xtick.major.width': 0.75,
    'ytick.major.width': 0.75,
    'lines.linewidth': 1.0,
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})


This section loads the current CMAscan inputs and rebuilds the canonical sequence-screen validation sets from the updated dataset.

In [ ]:
#@title Load CMAscan inputs

TYPE_CANONICAL = 'C'
DATASET_POSITIVE = 'P'
DATASET_PUTATIVE_POSITIVE = 'P*'
BIOLOGICAL_PROCESS = 'CMA'
NOT_INVESTIGATED = 'NI'
INVESTIGATION_COLUMNS = [
    'Interaction with LAMP2A',
    'Interaction with HSC70',
    'Association with lysosomes',
]


def load_cmascan_dataset(relative_path='dataset/cma_motif_dataset.csv'):
    dataset_path = get_repo_local_copy(relative_path)
    text = dataset_path.read_text(encoding='utf-8')
    text = text.replace('\\r\\n', '\n').replace('\\n', '\n').replace('\\r', '\n')
    df = pd.read_csv(StringIO(text), engine='python')
    df['UniProt ID'] = df['UniProt ID'].astype(str).str.extract(r'([A-Z0-9]{6})')
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    return df


motif_db_raw = load_cmascan_dataset()
motif_db = motif_db_raw[
    (motif_db_raw['Type'] == TYPE_CANONICAL) &
    (motif_db_raw['Biological process'] == BIOLOGICAL_PROCESS)
].copy()

structure_validation = pd.read_excel(get_repo_local_copy('dataset/motif_structures_sasa_v0.6.xlsx'))
structure_validation = structure_validation.rename(columns={
    'Uniprot ID': 'UniProt ID',
    'calculated_sasa': 'SASA',
    'IUPred3 score long mode': 'IUPred3',
})
structure_validation = structure_validation[structure_validation['Target'].isin(['Positive', 'Unlabeled'])].copy()
structure_validation['SASA_normalized'] = (
    (structure_validation['SASA'] - structure_validation['SASA'].min()) /
    (structure_validation['SASA'].max() - structure_validation['SASA'].min())
)

with open(get_repo_local_copy('PSSM/cPSSM.pkl'), 'rb') as handle:
    c_pssm = pickle.load(handle)
with open(get_repo_local_copy('PSSM/ePSSM.pkl'), 'rb') as handle:
    e_pssm = pickle.load(handle)

fasta_path = get_repo_local_copy('dataset/dataset_seqs.fasta')
protein_name_map = (
    motif_db[['UniProt ID', 'Protein']]
    .dropna(subset=['UniProt ID'])
    .drop_duplicates(subset=['UniProt ID'])
    .set_index('UniProt ID')['Protein']
    .to_dict()
)

print(f'Canonical CMA records in motif dataset: {len(motif_db)}')
print(f'Structure-validation rows: {len(structure_validation)}')
print(f'Unique proteins in structure-validation table: {structure_validation["UniProt ID"].nunique()}')
print(f'FASTA file: {fasta_path}')

display(motif_db.head())
display(structure_validation.head())


In [ ]:

#@title Define helper functions

def extract_uniprot_accession(header):
    match = re.search(r'\|([A-Z0-9]{6})\|', str(header))
    if match:
        return match.group(1)

    fallback = re.search(r'([A-Z0-9]{6})', str(header))
    return fallback.group(1) if fallback else None


def reorient_motif(motif):
    motif = str(motif).strip().upper()
    if motif and motif[0] not in {'Q', 'N'}:
        return motif[::-1]
    return motif


def calculate_pssm_score(motif, pssm_matrix):
    motif = reorient_motif(motif).replace('U', 'C')
    return sum(pssm_matrix[aa][idx] for idx, aa in enumerate(motif))


def is_canonical(motif):
    motif = str(motif).strip().upper()
    side = {'Q', 'N'}
    positive = {'K', 'R'}
    negative = {'D', 'E'}
    hydrophobic = {'F', 'L', 'I', 'V'}

    if len(motif) != 5:
        return False
    if any(residue not in side | positive | negative | hydrophobic for residue in motif):
        return False
    if not (motif[0] in side or motif[-1] in side):
        return False
    if sum(residue in side for residue in motif) != 1:
        return False
    if sum(residue in positive for residue in motif) not in {1, 2}:
        return False
    if sum(residue in negative for residue in motif) != 1:
        return False
    if sum(residue in hydrophobic for residue in motif) not in {1, 2}:
        return False
    return True


def find_unique_kmers(sequence, k=5):
    sequence = str(sequence).strip().upper()
    kmers = []
    for start in range(len(sequence) - k + 1):
        mer = sequence[start:start + k]
        if mer not in kmers:
            kmers.append(mer)
    return kmers


def expand_dataset_to_kmer_lookup(df_subset):
    rows = []
    for _, row in df_subset.iterrows():
        motif = str(row['Motif']).strip().upper()
        for mer in find_unique_kmers(motif, k=5):
            rows.append({
                'UniProt ID': row['UniProt ID'],
                'mer': mer,
                'curated_motif': motif,
                'curated_motif_reoriented': str(row['Motif reoriented']).strip().upper(),
                'curated_protein': row['Protein'],
                'DOI': row['DOI'],
                'Dataset': row['Dataset'],
            })

    lookup = pd.DataFrame(rows)
    if lookup.empty:
        return lookup

    return lookup.groupby(['UniProt ID', 'mer'], as_index=False).agg({
        'curated_motif': lambda s: '; '.join(sorted(set(map(str, s)))),
        'curated_motif_reoriented': lambda s: '; '.join(sorted(set(map(str, s)))),
        'curated_protein': lambda s: '; '.join(sorted(set(map(str, s)))),
        'DOI': lambda s: '; '.join(sorted(set(map(str, s)))),
        'Dataset': lambda s: '; '.join(sorted(set(map(str, s)))),
    })


def screen_fasta_for_canonical_motifs(fasta_file, c_pssm_matrix, e_pssm_matrix, protein_map):
    records = []

    for record in SeqIO.parse(str(fasta_file), 'fasta'):
        sequence = str(record.seq).upper()
        accession = extract_uniprot_accession(record.id)
        curated_protein = protein_map.get(accession, 'NA')

        for start in range(len(sequence) - 4):
            mer = sequence[start:start + 5]
            if not is_canonical(mer):
                continue

            records.append({
                'protein_name': record.id,
                'Protein': curated_protein,
                'UniProt ID': accession,
                'mer': mer,
                'mer_reoriented': reorient_motif(mer),
                'type': 'canonical',
                'localization': int(start + 1),
                'cPSSM': round(calculate_pssm_score(mer, c_pssm_matrix), 4),
                'ePSSM': round(calculate_pssm_score(mer, e_pssm_matrix), 4),
            })

    result = pd.DataFrame(records)
    if result.empty:
        return result

    return result.sort_values(['UniProt ID', 'localization', 'mer']).reset_index(drop=True)


def plot_group_score_comparison(
    df,
    group_col,
    score_columns,
    group_order,
    label_map,
    palette,
    width_mode='two_column',
    fig_height=None,
    dot_alpha=0.45,
    save_path=None,
    x_by='group',
):
    plot_df = df[[group_col] + score_columns].copy()
    plot_df = plot_df.melt(id_vars=group_col, var_name='Metric', value_name='Score')
    plot_df['Metric'] = plot_df['Metric'].map(label_map)
    plot_df[group_col] = pd.Categorical(plot_df[group_col], categories=group_order, ordered=True)

    fig_width, fig_height = get_publication_figsize(4.0, 3.0, width_mode=width_mode, fig_height=fig_height)

    sns.set(style='whitegrid')
    sns.set_context('notebook', font_scale=1.0)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    if x_by == 'metric':
        x_col = 'Metric'
        hue_col = group_col
        x_order = list(label_map.values())
        hue_order = group_order
    else:
        x_col = group_col
        hue_col = 'Metric'
        x_order = group_order
        hue_order = list(label_map.values())

    sns.boxplot(
        data=plot_df,
        x=x_col,
        y='Score',
        hue=hue_col,
        order=x_order,
        hue_order=hue_order,
        palette=palette,
        showfliers=False,
        linewidth=1.0,
        boxprops={'edgecolor': 'black', 'linewidth': 0.8},
        whiskerprops={'color': 'black', 'linewidth': 0.8},
        capprops={'color': 'black', 'linewidth': 0.8},
        medianprops={'color': 'black', 'linewidth': 0.9},
        ax=ax,
    )
    sns.stripplot(
        data=plot_df,
        x=x_col,
        y='Score',
        hue=hue_col,
        order=x_order,
        hue_order=hue_order,
        dodge=True,
        palette='dark:black',
        alpha=dot_alpha,
        size=2.6,
        ax=ax,
    )

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        n = len(hue_order)
        ax.legend(handles[:n], labels[:n], frameon=False, title=None, fontsize=9, handlelength=1.8)

    ax.set_xlabel('', fontsize=9)
    ax.set_ylabel('Score', fontsize=9)
    ax.tick_params(axis='both', which='both', labelsize=9, direction='out', length=3, width=0.8)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_linewidth(1.0)
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.4)
    fig.tight_layout()

    if save_path is not None:
        save_path = normalise_save_path(save_path, default_extension='svg')
        fig.savefig(save_path, format='svg', dpi=300, bbox_inches='tight', transparent=True)
        print(f'Figure saved to: {save_path}')

    plt.show()
    plt.close(fig)


def run_mannwhitney_table(df, group_col, group1, group2, score_columns, label_map):
    rows = []
    for column in score_columns:
        group1_values = pd.to_numeric(df.loc[df[group_col] == group1, column], errors='coerce').dropna().to_numpy()
        group2_values = pd.to_numeric(df.loc[df[group_col] == group2, column], errors='coerce').dropna().to_numpy()
        u_stat, p_value = stats.mannwhitneyu(group1_values, group2_values, alternative='greater')

        rows.append({
            'metric': label_map.get(column, column),
            'group_1': group1,
            'group_2': group2,
            'n_group_1': int(group1_values.size),
            'n_group_2': int(group2_values.size),
            'median_group_1': round(float(np.median(group1_values)), 4),
            'median_group_2': round(float(np.median(group2_values)), 4),
            'U_statistic': round(float(u_stat), 4),
            'p_value_one_sided': round(float(p_value), 6),
        })

    return pd.DataFrame(rows)


def fisher_above_threshold(pos_series, neg_series, threshold):
    pos_series = pd.to_numeric(pos_series, errors='coerce').dropna()
    neg_series = pd.to_numeric(neg_series, errors='coerce').dropna()

    pos_above = int((pos_series >= threshold).sum())
    pos_below = int((pos_series < threshold).sum())
    neg_above = int((neg_series >= threshold).sum())
    neg_below = int((neg_series < threshold).sum())

    contingency = [[pos_above, pos_below], [neg_above, neg_below]]
    odds_ratio, p_value = stats.fisher_exact(contingency, alternative='greater')

    return {
        'threshold': threshold,
        'group_1_above_or_equal': pos_above,
        'group_1_below': pos_below,
        'group_2_above_or_equal': neg_above,
        'group_2_below': neg_below,
        'odds_ratio': float(odds_ratio),
        'p_value_one_sided': float(p_value),
    }


def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))


def estimate_mannwhitney_power(u_statistic, n1, n2, alpha=0.05):
    auc = u_statistic / (n1 * n2)
    rank_biserial = abs((2 * auc) - 1)

    if rank_biserial >= 0.999999:
        effect_size = 10.0
    else:
        effect_size = 2 * rank_biserial / np.sqrt(max(1e-12, 1 - rank_biserial ** 2))

    power = TTestIndPower().power(
        effect_size=effect_size,
        nobs1=n1,
        ratio=n2 / n1,
        alpha=alpha,
        alternative='larger',
    )

    return rank_biserial, effect_size, power


def calculate_wilcoxon_power(pos_scores, neg_scores, alpha=0.05):
    diffs = np.asarray(pos_scores, dtype=float) - np.asarray(neg_scores, dtype=float)
    n_pairs = diffs.size
    std_diff = np.std(diffs, ddof=1)
    if std_diff == 0:
        effect_size = np.inf
        power = 1.0
    else:
        effect_size = float(np.mean(diffs) / std_diff)
        power = TTestPower().power(effect_size=effect_size, nobs=n_pairs, alpha=alpha, alternative='larger')
    return effect_size, power


def compare_groups_above_threshold(df, group_col, score_col, group1, group2, threshold=0.2, alpha=0.05):
    scores1 = pd.to_numeric(df.loc[df[group_col] == group1, score_col], errors='coerce').dropna()
    scores2 = pd.to_numeric(df.loc[df[group_col] == group2, score_col], errors='coerce').dropna()

    a1 = int((scores1 > threshold).sum())
    b1 = int((scores1 <= threshold).sum())
    a2 = int((scores2 > threshold).sum())
    b2 = int((scores2 <= threshold).sum())

    contingency = pd.DataFrame(
        [[a1, b1], [a2, b2]],
        index=[group1, group2],
        columns=[f'>{threshold}', f'≤{threshold}'],
    )

    _, fisher_p = stats.fisher_exact(contingency.values, alternative='greater')

    boschloo_callable = getattr(stats, 'boschloo_exact', None)
    boschloo_p = np.nan
    if boschloo_callable is not None:
        boschloo_p = float(boschloo_callable(contingency.values, alternative='greater').pvalue)

    p1 = a1 / (a1 + b1) if (a1 + b1) else np.nan
    p2 = a2 / (a2 + b2) if (a2 + b2) else np.nan

    if np.isnan(p1) or np.isnan(p2) or p1 == p2:
        power = np.nan
    else:
        effect_size = cohens_h(p1, p2)
        power = NormalIndPower().power(
            effect_size=effect_size,
            nobs1=a1 + b1,
            alpha=alpha,
            ratio=(a2 + b2) / (a1 + b1),
            alternative='larger',
        )

    return contingency, fisher_p, boschloo_p, power


In [ ]:

#@title Screen dataset proteins for canonical motifs

sequence_screen_results = screen_fasta_for_canonical_motifs(
    fasta_file=fasta_path,
    c_pssm_matrix=c_pssm,
    e_pssm_matrix=e_pssm,
    protein_map=protein_name_map,
)
sequence_screen_results.to_csv(RESULTS_DIR / 'validation_sequence_screen_results.csv', index=False)

print(f'Canonical hits found in dataset FASTA: {len(sequence_screen_results)}')
print(f'Proteins with at least one canonical hit: {sequence_screen_results["UniProt ID"].nunique()}')
print(f'Results saved to: {RESULTS_DIR / "validation_sequence_screen_results.csv"}')

display(sequence_screen_results.head())


In [ ]:

#@title Create validated positive motif set

positive_reference_db = motif_db[motif_db['Dataset'] == DATASET_POSITIVE].copy()
positive_lookup = expand_dataset_to_kmer_lookup(positive_reference_db)
positive_set = sequence_screen_results.merge(positive_lookup, on=['UniProt ID', 'mer'], how='inner')
positive_set = positive_set.sort_values(['UniProt ID', 'localization', 'mer']).reset_index(drop=True)
positive_set.to_csv(RESULTS_DIR / 'validation_positive_set.csv', index=False)

print(f'Positive records: {len(positive_set)}')
print(f'Unique positive motifs: {positive_set["mer"].nunique()}')
print(f'Positive proteins: {positive_set["UniProt ID"].nunique()}')
print(f'Results saved to: {RESULTS_DIR / "validation_positive_set.csv"}')

display(positive_set.head())


In [ ]:

#@title Create putative positive motif set

putative_positive_reference_db = motif_db[motif_db['Dataset'] == DATASET_PUTATIVE_POSITIVE].copy()
putative_positive_lookup = expand_dataset_to_kmer_lookup(putative_positive_reference_db)
putative_positive_set = sequence_screen_results.merge(putative_positive_lookup, on=['UniProt ID', 'mer'], how='inner')
putative_positive_set = putative_positive_set.sort_values(['UniProt ID', 'localization', 'mer']).reset_index(drop=True)
putative_positive_set.to_csv(RESULTS_DIR / 'validation_putative_positive_set.csv', index=False)

print(f'Putative-positive records: {len(putative_positive_set)}')
print(f'Unique putative-positive motifs: {putative_positive_set["mer"].nunique()}')
print(f'Putative-positive proteins: {putative_positive_set["UniProt ID"].nunique()}')
print(f'Results saved to: {RESULTS_DIR / "validation_putative_positive_set.csv"}')

display(putative_positive_set.head())


In [ ]:

#@title Create putative negative motif set

confirmed_positive_pairs = set(zip(positive_set['UniProt ID'], positive_set['mer']))
positive_proteins = set(positive_set['UniProt ID'])

putative_negative_set = sequence_screen_results[
    sequence_screen_results['UniProt ID'].isin(positive_proteins)
].copy()
putative_negative_set = putative_negative_set[
    ~putative_negative_set.apply(lambda row: (row['UniProt ID'], row['mer']) in confirmed_positive_pairs, axis=1)
].copy()
putative_negative_set = putative_negative_set.sort_values(['UniProt ID', 'localization', 'mer']).reset_index(drop=True)
putative_negative_set.to_csv(RESULTS_DIR / 'validation_putative_negative_set.csv', index=False)

print(f'Putative-negative records: {len(putative_negative_set)}')
print(f'Unique putative-negative motifs: {putative_negative_set["mer"].nunique()}')
print(f'Putative-negative proteins: {putative_negative_set["UniProt ID"].nunique()}')
print(f'Results saved to: {RESULTS_DIR / "validation_putative_negative_set.csv"}')

display(putative_negative_set.head())


In [ ]:

#@title Run sanity checks against the structure-validation table

sequence_positive_pairs = set(zip(positive_set['UniProt ID'], positive_set['mer']))
sequence_negative_pairs = set(zip(putative_negative_set['UniProt ID'], putative_negative_set['mer']))
structure_positive_pairs = set(zip(
    structure_validation.loc[structure_validation['Target'] == 'Positive', 'UniProt ID'],
    structure_validation.loc[structure_validation['Target'] == 'Positive', 'Motif'],
))
structure_negative_pairs = set(zip(
    structure_validation.loc[structure_validation['Target'] == 'Unlabeled', 'UniProt ID'],
    structure_validation.loc[structure_validation['Target'] == 'Unlabeled', 'Motif'],
))

assert sequence_positive_pairs == structure_positive_pairs, 'Positive motif pairs do not match the structure-validation table.'
assert sequence_negative_pairs == structure_negative_pairs, 'Putative-negative motif pairs do not match the structure-validation table.'

print('Sanity checks passed.')
print(f'Positive pairs matched: {len(sequence_positive_pairs)}')
print(f'Putative-negative pairs matched: {len(sequence_negative_pairs)}')


In [ ]:

#@title Export per-protein summary tables

positive_summary = (
    positive_set.groupby(['UniProt ID', 'Protein', 'protein_name'])['mer']
    .agg(list)
    .reset_index()
    .rename(columns={'mer': 'Positive motifs'})
)
positive_summary['Positive motifs count'] = positive_summary['Positive motifs'].apply(len)

negative_summary = (
    putative_negative_set.groupby(['UniProt ID', 'Protein', 'protein_name'])['mer']
    .agg(list)
    .reset_index()
    .rename(columns={'mer': 'Putative negative motifs'})
)
negative_summary['Putative negative motifs count'] = negative_summary['Putative negative motifs'].apply(len)

summary_positive_negative = positive_summary.merge(
    negative_summary,
    on=['UniProt ID', 'Protein', 'protein_name'],
    how='outer',
)
summary_positive_negative['Positive motifs'] = summary_positive_negative['Positive motifs'].apply(lambda value: value if isinstance(value, list) else [])
summary_positive_negative['Putative negative motifs'] = summary_positive_negative['Putative negative motifs'].apply(lambda value: value if isinstance(value, list) else [])
summary_positive_negative['Positive motifs count'] = summary_positive_negative['Positive motifs count'].fillna(0).astype(int)
summary_positive_negative['Putative negative motifs count'] = summary_positive_negative['Putative negative motifs count'].fillna(0).astype(int)
summary_positive_negative = summary_positive_negative.sort_values(['UniProt ID', 'Protein']).reset_index(drop=True)
summary_positive_negative.to_csv(RESULTS_DIR / 'validation_summary_positive_negative_per_protein.csv', index=False)

summary_putative_positive = (
    putative_positive_set.groupby(['UniProt ID', 'Protein', 'protein_name'])['mer']
    .agg(list)
    .reset_index()
    .rename(columns={'mer': 'Putative positive motifs'})
)
summary_putative_positive['Putative positive motifs count'] = summary_putative_positive['Putative positive motifs'].apply(len)
summary_putative_positive = summary_putative_positive.sort_values(['UniProt ID', 'Protein']).reset_index(drop=True)
summary_putative_positive.to_csv(RESULTS_DIR / 'validation_summary_putative_positive_per_protein.csv', index=False)

print(f'Summary 1 saved to: {RESULTS_DIR / "validation_summary_positive_negative_per_protein.csv"}')
print(f'Summary 2 saved to: {RESULTS_DIR / "validation_summary_putative_positive_per_protein.csv"}')

display(summary_positive_negative.head())
display(summary_putative_positive.head())


In [ ]:

#@title Plot cPSSM and ePSSM comparison

plot_group_score_comparison(
    df=pd.concat([
        putative_positive_set.assign(Set='Putative positive'),
        putative_negative_set.assign(Set='Putative negative'),
    ], ignore_index=True),
    group_col='Set',
    score_columns=['cPSSM', 'ePSSM'],
    group_order=['Putative positive', 'Putative negative'],
    label_map={'cPSSM': 'cPSSM', 'ePSSM': 'ePSSM'},
    palette={'cPSSM': '#0072B2', 'ePSSM': '#D55E00'},
    width_mode='three_column',
    fig_height=3.0,
    save_path=FIGURES_DIR / 'validation_pssm_comparison.svg',
)


In [ ]:

#@title Compare putative-positive and putative-negative PSSM scores

pssm_mannwhitney = run_mannwhitney_table(
    df=pd.concat([
        putative_positive_set.assign(Set='Putative positive'),
        putative_negative_set.assign(Set='Putative negative'),
    ], ignore_index=True),
    group_col='Set',
    group1='Putative positive',
    group2='Putative negative',
    score_columns=['cPSSM', 'ePSSM'],
    label_map={'cPSSM': 'cPSSM', 'ePSSM': 'ePSSM'},
)
pssm_mannwhitney.to_csv(RESULTS_DIR / 'validation_pssm_mannwhitney.csv', index=False)

print(f'Results saved to: {RESULTS_DIR / "validation_pssm_mannwhitney.csv"}')
display(pssm_mannwhitney)


In [ ]:

#@title Compare proportions above cPSSM and ePSSM thresholds

threshold_rows = []
for metric, threshold in [('cPSSM', 5.0), ('ePSSM', 5.0)]:
    fisher_result = fisher_above_threshold(
        putative_positive_set[metric],
        putative_negative_set[metric],
        threshold=threshold,
    )
    fisher_result['metric'] = metric
    threshold_rows.append(fisher_result)

pssm_thresholds = pd.DataFrame(threshold_rows)[[
    'metric',
    'threshold',
    'group_1_above_or_equal',
    'group_1_below',
    'group_2_above_or_equal',
    'group_2_below',
    'odds_ratio',
    'p_value_one_sided',
]]
pssm_thresholds.to_csv(RESULTS_DIR / 'validation_pssm_fisher_thresholds.csv', index=False)

print(f'Results saved to: {RESULTS_DIR / "validation_pssm_fisher_thresholds.csv"}')
display(pssm_thresholds)


The structure-validation section keeps only the current CMAscan features that are still in use: normalized SASA and IUPred3 long-mode scores.

In [ ]:

#@title Prepare the structure-validation table

structure_validation_export = structure_validation[[
    'Motif',
    'Protein',
    'UniProt ID',
    'Total protein length',
    'Motif localization',
    'IUPred3',
    'SASA',
    'SASA_normalized',
    'Target',
]].copy()
structure_validation_export.to_csv(RESULTS_DIR / 'validation_structure_table.csv', index=False)

print(f'Positive rows: {(structure_validation_export["Target"] == "Positive").sum()}')
print(f'Unlabeled rows: {(structure_validation_export["Target"] == "Unlabeled").sum()}')
print(f'Results saved to: {RESULTS_DIR / "validation_structure_table.csv"}')

display(structure_validation_export.head())


In [ ]:

#@title Plot normalized SASA and IUPred3 comparison

plot_group_score_comparison(
    df=structure_validation_export.rename(columns={'Target': 'Set'}),
    group_col='Set',
    score_columns=['SASA_normalized', 'IUPred3'],
    group_order=['Positive', 'Unlabeled'],
    label_map={'SASA_normalized': 'SASA', 'IUPred3': 'IUPred3'},
    palette={'Positive': '#0072B2', 'Unlabeled': '#D55E00'},
    width_mode='three_column',
    fig_height=None,
    save_path=FIGURES_DIR / 'validation_structure_scores.svg',
    x_by='metric',
)


In [ ]:

#@title Compare positive and unlabeled structure scores

structure_mannwhitney = run_mannwhitney_table(
    df=structure_validation_export.rename(columns={'Target': 'Set'}),
    group_col='Set',
    group1='Positive',
    group2='Unlabeled',
    score_columns=['SASA_normalized', 'IUPred3'],
    label_map={'SASA_normalized': 'SASA', 'IUPred3': 'IUPred3'},
)
structure_mannwhitney.to_csv(RESULTS_DIR / 'validation_structure_mannwhitney.csv', index=False)

print(f'Results saved to: {RESULTS_DIR / "validation_structure_mannwhitney.csv"}')
display(structure_mannwhitney)


In [ ]:

#@title Estimate power for Mann–Whitney comparisons

power_rows = []
for _, row in pssm_mannwhitney.iterrows():
    rank_biserial, effect_size, power = estimate_mannwhitney_power(
        u_statistic=row['U_statistic'],
        n1=int(row['n_group_1']),
        n2=int(row['n_group_2']),
    )
    power_rows.append({
        'metric': row['metric'],
        'comparison': 'Putative positive vs Putative negative',
        'rank_biserial': round(float(rank_biserial), 4),
        'effect_size_approx': round(float(effect_size), 4),
        'power': round(float(power), 4),
    })

for _, row in structure_mannwhitney.iterrows():
    rank_biserial, effect_size, power = estimate_mannwhitney_power(
        u_statistic=row['U_statistic'],
        n1=int(row['n_group_1']),
        n2=int(row['n_group_2']),
    )
    power_rows.append({
        'metric': row['metric'],
        'comparison': 'Positive vs Unlabeled',
        'rank_biserial': round(float(rank_biserial), 4),
        'effect_size_approx': round(float(effect_size), 4),
        'power': round(float(power), 4),
    })

mannwhitney_power = pd.DataFrame(power_rows)
mannwhitney_power.to_csv(RESULTS_DIR / 'validation_mannwhitney_power.csv', index=False)

print(f'Results saved to: {RESULTS_DIR / "validation_mannwhitney_power.csv"}')
display(mannwhitney_power)


In [ ]:

#@title Run paired Wilcoxon tests on per-protein median structure scores

paired_rows = []
protein_counts = structure_validation_export.groupby(['UniProt ID', 'Target']).size().unstack(fill_value=0)

for uniprot_id in protein_counts.index:
    if protein_counts.loc[uniprot_id].get('Positive', 0) == 0:
        continue
    if protein_counts.loc[uniprot_id].get('Unlabeled', 0) == 0:
        continue

    positive_block = structure_validation_export[
        (structure_validation_export['UniProt ID'] == uniprot_id) &
        (structure_validation_export['Target'] == 'Positive')
    ]
    negative_block = structure_validation_export[
        (structure_validation_export['UniProt ID'] == uniprot_id) &
        (structure_validation_export['Target'] == 'Unlabeled')
    ]

    paired_rows.append({
        'UniProt ID': uniprot_id,
        'Protein': positive_block['Protein'].iloc[0],
        'positive_SASA_median': float(positive_block['SASA_normalized'].median()),
        'unlabeled_SASA_median': float(negative_block['SASA_normalized'].median()),
        'positive_IUPred3_median': float(positive_block['IUPred3'].median()),
        'unlabeled_IUPred3_median': float(negative_block['IUPred3'].median()),
    })

paired_structure_medians = pd.DataFrame(paired_rows).sort_values(['UniProt ID', 'Protein']).reset_index(drop=True)
paired_structure_medians.to_csv(RESULTS_DIR / 'validation_paired_structure_medians.csv', index=False)

wilcoxon_rows = []
for positive_col, negative_col, label in [
    ('positive_SASA_median', 'unlabeled_SASA_median', 'SASA'),
    ('positive_IUPred3_median', 'unlabeled_IUPred3_median', 'IUPred3'),
]:
    statistic, p_value = stats.wilcoxon(
        paired_structure_medians[positive_col],
        paired_structure_medians[negative_col],
        alternative='greater',
    )
    wilcoxon_rows.append({
        'metric': label,
        'n_pairs': int(len(paired_structure_medians)),
        'positive_median': round(float(np.median(paired_structure_medians[positive_col])), 4),
        'unlabeled_median': round(float(np.median(paired_structure_medians[negative_col])), 4),
        'wilcoxon_statistic': round(float(statistic), 4),
        'p_value_one_sided': round(float(p_value), 6),
    })

wilcoxon_results = pd.DataFrame(wilcoxon_rows)
wilcoxon_results.to_csv(RESULTS_DIR / 'validation_paired_wilcoxon.csv', index=False)

print(f'Paired medians saved to: {RESULTS_DIR / "validation_paired_structure_medians.csv"}')
print(f'Test results saved to: {RESULTS_DIR / "validation_paired_wilcoxon.csv"}')
display(paired_structure_medians.head())
display(wilcoxon_results)


In [ ]:

#@title Estimate power for paired Wilcoxon tests

paired_power_rows = []
for positive_col, negative_col, label in [
    ('positive_SASA_median', 'unlabeled_SASA_median', 'SASA'),
    ('positive_IUPred3_median', 'unlabeled_IUPred3_median', 'IUPred3'),
]:
    effect_size, power = calculate_wilcoxon_power(
        paired_structure_medians[positive_col],
        paired_structure_medians[negative_col],
    )
    paired_power_rows.append({
        'metric': label,
        'n_pairs': int(len(paired_structure_medians)),
        'effect_size_dz': round(float(effect_size), 4) if np.isfinite(effect_size) else effect_size,
        'power': round(float(power), 4),
    })

paired_power = pd.DataFrame(paired_power_rows)
paired_power.to_csv(RESULTS_DIR / 'validation_paired_wilcoxon_power.csv', index=False)

print(f'Results saved to: {RESULTS_DIR / "validation_paired_wilcoxon_power.csv"}')
display(paired_power)


In [ ]:

#@title Run Fisher exact test for IUPred3 above 0.2

contingency_table, fisher_p_value, _, threshold_power = compare_groups_above_threshold(
    df=structure_validation_export,
    group_col='Target',
    score_col='IUPred3',
    group1='Positive',
    group2='Unlabeled',
    threshold=0.2,
)

iupred_fisher = contingency_table.reset_index().rename(columns={'index': 'Group'})
iupred_fisher['fisher_p_value_one_sided'] = ''
iupred_fisher['power'] = ''
iupred_fisher.loc[0, 'fisher_p_value_one_sided'] = round(float(fisher_p_value), 6)
iupred_fisher.loc[0, 'power'] = round(float(threshold_power), 4) if not np.isnan(threshold_power) else np.nan
iupred_fisher.to_csv(RESULTS_DIR / 'validation_iupred_fisher_threshold.csv', index=False)

print(f'Results saved to: {RESULTS_DIR / "validation_iupred_fisher_threshold.csv"}')
display(iupred_fisher)


In [ ]:

#@title Run Boschloo exact test for IUPred3 above 0.2

contingency_table, _, boschloo_p_value, threshold_power = compare_groups_above_threshold(
    df=structure_validation_export,
    group_col='Target',
    score_col='IUPred3',
    group1='Positive',
    group2='Unlabeled',
    threshold=0.2,
)

iupred_boschloo = contingency_table.reset_index().rename(columns={'index': 'Group'})
iupred_boschloo['boschloo_p_value_one_sided'] = ''
iupred_boschloo['power'] = ''
iupred_boschloo.loc[0, 'boschloo_p_value_one_sided'] = round(float(boschloo_p_value), 6) if not np.isnan(boschloo_p_value) else np.nan
iupred_boschloo.loc[0, 'power'] = round(float(threshold_power), 4) if not np.isnan(threshold_power) else np.nan
iupred_boschloo.to_csv(RESULTS_DIR / 'validation_iupred_boschloo_threshold.csv', index=False)

print(f'Results saved to: {RESULTS_DIR / "validation_iupred_boschloo_threshold.csv"}')
display(iupred_boschloo)
